# Vision Module: Intrinsic Condition Encoder
## Two-Stage Architecture: Identity Disentanglement + Condition Residual

**Core question:** Can a CNN trained on PSA slab images learn a latent condition embedding whose internal structure correlates with established grading dimensions and whose visual attributions localise to condition-relevant regions of the card?

**Design requirement:** The framework shall include a CNN-based intrinsic encoder that takes high-resolution PSA slab images of Pokémon cards in grades 8-10 and outputs a 256-dimensional latent condition embedding capturing centering, corners, edge wear, surface quality, and print defects.

**Design requirement:** The framework shall preserve partial separability during training through branch-specific auxiliary objectives designed to maintain representational separation.


### Design rationale: Why two-stage?

Locatello et al. (2019) established that disentanglement does not emerge automatically without inductive bias. The two-stage design provides that bias:

- **Stage 1 (Card identity encoder):** Learns to classify card_name (7 classes). This forces the embedding to absorb set/artwork/era variation. Frozen after training.
- **Stage 2 (Condition residual encoder):** Learns to classify grade (3 classes) with an orthogonality constraint against the Stage 1 identity embedding. This forces the condition embedding to capture the physical condition that Stage 1 does not.

The orthogonality constraint operationalises the separation-preserving design goal at the representation level. If Stage 2 still cannot separate grades after identity is controlled for, that is itself an informative negative result. It indicates that PSA 8-10 condition differences are not reliably visible at ImageNet feature resolution, which is itself a valid project finding.


### Notebook structure

| Section | Content
|---------|---------
| 0 | Setup and configuration
| 1 | Data loading and integrity checks
| 2 | Preprocessing and augmentation
| 3 | Stratified split construction
| 4 | Stage 1: Card identity encoder
| 5 | Stage 1: Training and evaluation
| 6 | Stage 2: Condition residual encoder
| 7 | Stage 2: Training and evaluation
| 8 | Embedding extraction
| 9 | Representation quality assessment
| 10 | Economic relevance (embedding-price)
| 11 | Shortcut detection
| 13 | Failure mode documentation
| 14 | Final assessment


## Section 0: Setup and configuration


In [ ]:
## Project root — works locally or in Google Colab
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/pokemon-card-valuation')
except ImportError:
    ## Local / non-Colab: assume this notebook runs from the repo's notebooks/ folder
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()


In [ ]:
## Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from torch.nn.functional import adaptive_avg_pool2d
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
import json
import time
import random
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from torch.nn.functional import adaptive_avg_pool2d

import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
## Reproducibility
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print(f'Seed set: {SEED}')

In [ ]:
## Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if device.type == 'cuda':
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
## Path configuration and master config dictionary

CONFIG = {
    ## Paths
    'data_dir': PROJECT_ROOT / 'data/processed',
    'image_dir': PROJECT_ROOT / 'data/raw/images/final',
    'model_dir': PROJECT_ROOT / 'models/vision_v2',
    'results_dir': PROJECT_ROOT / 'results/vision_v2',
    'figures_dir': PROJECT_ROOT / 'results/vision_v2/figures',
    'embeddings_dir': PROJECT_ROOT / 'data/embeddings',

    ## Data
    'cards_file': 'cards_clean_v2.parquet',
    'prices_file': 'prices_clean_v2.parquet',
    'seed': SEED,

    ## Image preprocessing
    'image_size': 224,
    'imagenet_mean': [0.485, 0.456, 0.406],
    'imagenet_std': [0.229, 0.224, 0.225],

    ## Architecture
    'backbone': 'resnet50',
    'backbone_frozen': True,
    'embedding_dim': 256,
    'dropout': 0.3,

    ## Stage 1: Card Identity Encoder
    'stage1': {
        'num_classes': 7,           ## 7 Pokémon
        'task': 'card_name',
        'epochs': 30,
        'batch_size': 32,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'patience': 7,
        'label_smoothing': 0.0,     ## Clean labels for identity
    },

    # Stage 2: Condition Residual Encoder
    'stage2': {
        'num_classes': 3,           ## PSA 8, 9, 10
        'task': 'grade',
        'epochs': 30,
        'batch_size': 32,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'patience': 7,
        'label_smoothing': 0.1,     ## Noisy supervision for grades
        'ortho_weight': 0.1,        ## Orthogonality penalty weight
    },

    ## Split
    'train_ratio': 0.70,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
}

## Create output directories
for d in ['model_dir', 'results_dir', 'figures_dir', 'embeddings_dir']:
    CONFIG[d].mkdir(parents=True, exist_ok=True)

## Verify data paths exist
cards_path = CONFIG['data_dir'] / CONFIG['cards_file']
prices_path = CONFIG['data_dir'] / CONFIG['prices_file']

assert cards_path.exists(), f'Cards file not found: {cards_path}'
assert prices_path.exists(), f'Prices file not found: {prices_path}'
assert CONFIG['image_dir'].exists(), f'Image dir not found: {CONFIG["image_dir"]}'

print('CONFIGURATION')
print(f'Data: {CONFIG["data_dir"]}')
print(f'Images: {CONFIG["image_dir"]}')
print(f'Models: {CONFIG["model_dir"]}')
print(f'Results: {CONFIG["results_dir"]}')
print(f'\nBackbone: {CONFIG["backbone"]} (frozen: {CONFIG["backbone_frozen"]})')
print(f'Embedding dim: {CONFIG["embedding_dim"]}')
print(f'\nStage 1: {CONFIG["stage1"]["task"]} ({CONFIG["stage1"]["num_classes"]} classes)')
print(f'Epochs: {CONFIG["stage1"]["epochs"]}, BS: {CONFIG["stage1"]["batch_size"]}, LR: {CONFIG["stage1"]["lr"]}')
print(f'Label smoothing: {CONFIG["stage1"]["label_smoothing"]}')
print(f'\nStage 2: {CONFIG["stage2"]["task"]} ({CONFIG["stage2"]["num_classes"]} classes)')
print(f'Epochs: {CONFIG["stage2"]["epochs"]}, BS: {CONFIG["stage2"]["batch_size"]}, LR: {CONFIG["stage2"]["lr"]}')
print(f'Label smoothing: {CONFIG["stage2"]["label_smoothing"]}')
print(f'Orthogonality weight: {CONFIG["stage2"]["ortho_weight"]}')
print(f'\nSplit: {CONFIG["train_ratio"]}/{CONFIG["val_ratio"]}/{CONFIG["test_ratio"]}')
print(f'Seed: {CONFIG["seed"]}')

In [ ]:
## Save config for reproducibility
config_save = {
    k: str(v) if isinstance(v, Path) else v
    for k, v in CONFIG.items()
}
with open(CONFIG['results_dir'] / 'vision_v2_config.json', 'w') as f:
    json.dump(config_save, f, indent=2, default=str)

print('Config saved to results/vision_v2/vision_v2_config.json')

## Section 1: Data loading and integrity checks

In [ ]:
## Load cleaned datasets
cards = pd.read_parquet(CONFIG['data_dir'] / CONFIG['cards_file'])
prices = pd.read_parquet(CONFIG['data_dir'] / CONFIG['prices_file'])

print(f'Cards: {cards.shape}')
print(f'Prices: {prices.shape}')
print(f'\nCards columns: {list(cards.columns)}')
print(f'Prices columns: {list(prices.columns)}')

In [ ]:
## Verify grade and card_name distributions
print('GRADE DISTRIBUTION')
for grade, count in cards['grade'].value_counts().sort_index().items():
    print(f'PSA {grade}: {count:5d} ({count/len(cards)*100:.1f}%)')

print(f'\nCARD NAME DISTRIBUTION')
for name, count in cards['card_name'].value_counts().sort_index().items():
    print(f'{name:12s}: {count:5d} ({count/len(cards)*100:.1f}%)')

print(f'\nCARD+GRADE GROUPS')
cg = cards.groupby(['card_name', 'grade']).size().unstack(fill_value=0)
print(cg.to_string())
print(f'\nAll groups >= 5: {(cg.values.flatten() >= 5).all()}')

In [ ]:
## Verify image availability: sample check + full count
image_dir = CONFIG['image_dir']

missing_images = []
corrupt_images = []

for idx, row in cards.iterrows():
    fp = PROJECT_ROOT / row['local_image_path']
    if not fp.exists():
        missing_images.append(row['listing_id'])
    elif fp.stat().st_size < 1000:
        corrupt_images.append(row['listing_id'])

print(f'Total cards: {len(cards)}')
print(f'Missing images: {len(missing_images)}')
print(f'Corrupt/tiny images: {len(corrupt_images)}')
print(f'Valid images: {len(cards) - len(missing_images) - len(corrupt_images)}')

assert len(missing_images) == 0, f'Missing images: {missing_images[:5]}'
assert len(corrupt_images) == 0, f'Corrupt images: {corrupt_images[:5]}'
print('\nall images present and valid')

In [ ]:
## Verify listing_id uniqueness and match between cards and prices
cards_ids = set(cards['listing_id'])
prices_ids = set(prices['listing_id'])

print(f'Unique listing_ids in cards: {len(cards_ids)}')
print(f'Unique listing_ids in prices: {len(prices_ids)}')
print(f'Overlap: {len(cards_ids & prices_ids)}')
print(f'In cards but not prices: {len(cards_ids - prices_ids)}')
print(f'In prices but not cards: {len(prices_ids - cards_ids)}')

assert cards_ids == prices_ids, 'listing_id mismatch between cards and prices!'
assert cards['listing_id'].nunique() == len(cards), 'Duplicate listing_ids in cards!'
print('\nlisting_ids match and are unique')

In [ ]:
## Verify no vision-irrelevant data leaks into this module
## Vision module uses: listing_id, card_name, grade, local_image_path
## It must not use: price, date_sold, market features

vision_cols = ['listing_id', 'card_name', 'grade', 'local_image_path']
available = [c for c in vision_cols if c in cards.columns]
missing_cols = [c for c in vision_cols if c not in cards.columns]

print('Required columns for vision module:')
for c in vision_cols:
    status = 'Present' if c in cards.columns else 'Missing'
    print(f'{c}: {status}')

assert len(missing_cols) == 0, f'Missing required columns: {missing_cols}'

## Build vision-only dataframe
df_vision = cards[vision_cols].copy()

## Merge price for later economic relevance analysis (Section 10) stored but not used in training
df_vision = df_vision.merge(prices[['listing_id', 'price']], on='listing_id', how='left')

print(f'\nVision dataframe: {df_vision.shape}')
print(f'Columns: {list(df_vision.columns)}')
print(f'Null check:')
print(df_vision.isnull().sum().to_string())

print(f'\nvision dataframe constructed, price stored for evaluation only')

In [ ]:
## Display sample images, one per grade to visually verify data
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, grade in enumerate([8, 9, 10]):
    sample = df_vision[df_vision['grade'] == grade].iloc[0]
    img_path = PROJECT_ROOT / sample['local_image_path']
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f"PSA {grade} — {sample['card_name']}\n${sample['price']:.2f}")
    axes[i].axis('off')

plt.suptitle('Sample Images by Grade', fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'sample_images_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()
print('sample_images_by_grade.png')

## Section 2: Preprocessing and augmentation


In [ ]:
## Define transforms
## Training: augmentation to handle real-world image variation
## (perspective, lighting, rotation observed in sample images)
## Validation/Test: deterministic resize and normalize only

train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  ## slight position shift
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG['imagenet_mean'], std=CONFIG['imagenet_std']),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG['imagenet_mean'], std=CONFIG['imagenet_std']),
])

print('Train transforms:')
for t in train_transform.transforms:
    print(f'{t}')
print(f'\nEval transforms:')
for t in eval_transform.transforms:
    print(f'{t}')

In [ ]:
## Dataset class that supports both card_name and grade targets

class CardDataset(Dataset):
    """PSA card image dataset.

    Supports two target modes:
      - 'card_name': 7-class identity classification (Stage 1)
      - 'grade': 3-class condition classification (Stage 2)
    """

    def __init__(self, dataframe, target_col, label_map, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.target_col = target_col
        self.label_map = label_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        ## Load image
        img_path = PROJECT_ROOT / row['local_image_path']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        ## Target
        label = self.label_map[row[self.target_col]]

        return image, label, idx  ## idx for embedding extraction

## Label maps
CARD_NAME_MAP = {name: i for i, name in enumerate(sorted(cards['card_name'].unique()))}
GRADE_MAP = {8: 0, 9: 1, 10: 2}

## Inverse maps for display
CARD_NAME_INV = {v: k for k, v in CARD_NAME_MAP.items()}
GRADE_INV = {0: 8, 1: 9, 2: 10}

print('Card name label map:')
for name, idx in CARD_NAME_MAP.items():
    print(f'{name}: {idx}')
print(f'\nGrade label map: {GRADE_MAP}')

In [ ]:
## Load test dataset to verify that batch loads correctly
test_ds = CardDataset(df_vision.head(8), target_col='card_name',
                      label_map=CARD_NAME_MAP, transform=eval_transform)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False)

batch_imgs, batch_labels, batch_idxs = next(iter(test_loader))
print(f'Batch images shape: {batch_imgs.shape}')  ## [4, 3, 224, 224]
print(f'Batch labels: {batch_labels}')
print(f'Batch indices: {batch_idxs}')
print(f'Pixel range: [{batch_imgs.min():.3f}, {batch_imgs.max():.3f}]')

assert batch_imgs.shape == (4, 3, 224, 224), f'Wrong shape: {batch_imgs.shape}'

In [ ]:
## Visualise augmented vs original for sanity check
sample_row = df_vision.iloc[0]
img_path = PROJECT_ROOT / sample_row['local_image_path']
raw_img = Image.open(img_path).convert('RGB')

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(raw_img)
axes[0].set_title('Original')
axes[0].axis('off')

## Show 4 augmented versions (denormalize for display)
inv_normalize = transforms.Normalize(
    mean=[-m/s for m, s in zip(CONFIG['imagenet_mean'], CONFIG['imagenet_std'])],
    std=[1/s for s in CONFIG['imagenet_std']]
)

for i in range(4):
    aug_tensor = train_transform(raw_img)
    display_tensor = inv_normalize(aug_tensor)
    display_img = display_tensor.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[i+1].imshow(display_img)
    axes[i+1].set_title(f'Augmented {i+1}')
    axes[i+1].axis('off')

plt.suptitle(f"{sample_row['card_name']} PSA {sample_row['grade']}", fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## Image format distribution check
extensions = df_vision['local_image_path'].apply(lambda x: Path(x).suffix.lower())
print('Image format distribution:')
print(extensions.value_counts().to_string())

## Section 3: Stratified split construction

**Design decision:** Stratified random split for vision module. Images are time-invariant. A card's physical condition does not change across sale dates. Temporal splitting is enforced in the market and fusion modules where it matters. This decision was validated in v1 and remains justified.

Stratification is on `card_name × grade` (21 groups) to ensure all groups are represented in all splits.

In [ ]:
## Create stratification key and perform split
df_vision['strat_key'] = df_vision['card_name'] + '_' + df_vision['grade'].astype(str)

## First split: train vs (val+test)
train_df, temp_df = train_test_split(
    df_vision, test_size=(CONFIG['val_ratio'] + CONFIG['test_ratio']),
    stratify=df_vision['strat_key'], random_state=SEED
)

## Second split: val vs test (50/50 of remaining)
val_df, test_df = train_test_split(
    temp_df, test_size=CONFIG['test_ratio'] / (CONFIG['val_ratio'] + CONFIG['test_ratio']),
    stratify=temp_df['strat_key'], random_state=SEED
)

print(f'Train: {len(train_df)} ({len(train_df)/len(df_vision)*100:.1f}%)')
print(f'Val:   {len(val_df)} ({len(val_df)/len(df_vision)*100:.1f}%)')
print(f'Test:  {len(test_df)} ({len(test_df)/len(df_vision)*100:.1f}%)')
print(f'Total: {len(train_df) + len(val_df) + len(test_df)}')

In [ ]:
## Verify that there is no listing_id leakage across splits
train_ids = set(train_df['listing_id'])
val_ids = set(val_df['listing_id'])
test_ids = set(test_df['listing_id'])

assert len(train_ids & val_ids) == 0, 'Train-Val overlap!'
assert len(train_ids & test_ids) == 0, 'Train-Test overlap!'
assert len(val_ids & test_ids) == 0, 'Val-Test overlap!'
print('No listing_id leakage across splits')

In [ ]:
## Verify stratification, grade and card_name distributions across splits
print('GRADE DISTRIBUTION BY SPLIT')
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = split_df['grade'].value_counts().sort_index()
    pct = dist / len(split_df) * 100
    print(f'{split_name:5s}: PSA8={pct.get(8,0):5.1f}%  PSA9={pct.get(9,0):5.1f}%  PSA10={pct.get(10,0):5.1f}%')

print(f'\nCARD NAME DISTRIBUTION BY SPLIT')
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = split_df['card_name'].value_counts().sort_index()
    pct = dist / len(split_df) * 100
    line = '  '.join([f'{n[:4]}={p:.1f}%' for n, p in pct.items()])
    print(f'{split_name:5s}: {line}')

In [ ]:
## Save split assignments for reproducibility and downstream modules
train_df = train_df.assign(split='train')
val_df = val_df.assign(split='val')
test_df = test_df.assign(split='test')

## Store split IDs
split_ids = {
    'train': train_df['listing_id'].tolist(),
    'val': val_df['listing_id'].tolist(),
    'test': test_df['listing_id'].tolist(),
}

with open(CONFIG['results_dir'] / 'vision_v2_split_ids.json', 'w') as f:
    json.dump(split_ids, f)

print(f'Split IDs saved: train={len(split_ids["train"])}, val={len(split_ids["val"])}, test={len(split_ids["test"])}')

## Recombine for reference
df_all = pd.concat([train_df, val_df, test_df], ignore_index=True)


## Section 4: Stage 1 Card identity encoder


**Objective:** Train a classifier to predict card_name (7 classes) from slab images. This forces the embedding layer to absorb card identity (set, artwork, era, language). After training, the entire Stage 1 model is frozen. Its 256-dim embedding becomes the identity representation that Stage 2 must be orthogonal to.

**Design justification:** Locatello et al. (2019) showed that disentanglement requires inductive bias. By explicitly learning card identity first, we create a reference subspace. Stage 2's orthogonality constraint then forces the condition embedding into the complementary subspace.

In [ ]:
## Stage 1 Model Architecture

class IdentityEncoder(nn.Module):
    """Stage 1: Card identity encoder.

    Frozen ResNet50 backbone -> 256-dim identity embedding -> 7-class classifier.
    The embedding layer captures card identity (set, artwork, era).
    """

    def __init__(self, embedding_dim=256, num_classes=7, dropout=0.3):
        super().__init__()

        ## Backbone: ResNet50 pretrained, fully frozen
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])  ## Remove FC

        ## Freeze entire backbone
        for param in self.features.parameters():
            param.requires_grad = False

        ## Embedding head (trainable)
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, embedding_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        ## Classifier head (trainable)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            features = self.features(x)
        emb = self.embedding(features)
        logits = self.classifier(emb)
        return logits, emb

    def get_embedding(self, x):
        """Extract identity embedding only (no classifier)."""
        with torch.no_grad():
            features = self.features(x)
            emb = self.embedding(features)
        return emb

print('IdentityEncoder defined')

In [ ]:
## Instantiate and verify parameter counts
stage1_model = IdentityEncoder(
    embedding_dim=CONFIG['embedding_dim'],
    num_classes=CONFIG['stage1']['num_classes'],
    dropout=CONFIG['dropout']
).to(device)

total_params = sum(p.numel() for p in stage1_model.parameters())
trainable_params = sum(p.numel() for p in stage1_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable_params:>12,}')
print(f'Frozen parameters:    {frozen_params:>12,}')
print(f'Trainable ratio:      {trainable_params/total_params*100:.2f}%')

## Verify backbone is frozen
backbone_trainable = sum(p.numel() for p in stage1_model.features.parameters() if p.requires_grad)
assert backbone_trainable == 0, f'Backbone has {backbone_trainable} trainable params!'
print(f'\nBackbone trainable params: {backbone_trainable} (confirmed frozen)')

In [ ]:
## Create dataloaders for Stage 1 (card_name classification)
train_dataset_s1 = CardDataset(train_df, target_col='card_name',
                               label_map=CARD_NAME_MAP, transform=train_transform)
val_dataset_s1 = CardDataset(val_df, target_col='card_name',
                             label_map=CARD_NAME_MAP, transform=eval_transform)
test_dataset_s1 = CardDataset(test_df, target_col='card_name',
                              label_map=CARD_NAME_MAP, transform=eval_transform)

train_loader_s1 = DataLoader(train_dataset_s1, batch_size=CONFIG['stage1']['batch_size'],
                             shuffle=True, num_workers=2, pin_memory=True)
val_loader_s1 = DataLoader(val_dataset_s1, batch_size=CONFIG['stage1']['batch_size'],
                           shuffle=False, num_workers=2, pin_memory=True)
test_loader_s1 = DataLoader(test_dataset_s1, batch_size=CONFIG['stage1']['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)

print(f'Stage 1 dataloaders:')
print(f'Train: {len(train_dataset_s1)} samples, {len(train_loader_s1)} batches')
print(f'Val:   {len(val_dataset_s1)} samples, {len(val_loader_s1)} batches')
print(f'Test:  {len(test_dataset_s1)} samples, {len(test_loader_s1)} batches')

In [ ]:
## Loss, optimizer, scheduler for Stage 1
criterion_s1 = nn.CrossEntropyLoss(
    label_smoothing=CONFIG['stage1']['label_smoothing']  # 0.0, clean labels
)

optimizer_s1 = optim.Adam(
    filter(lambda p: p.requires_grad, stage1_model.parameters()),
    lr=CONFIG['stage1']['lr'],
    weight_decay=CONFIG['stage1']['weight_decay']
)

scheduler_s1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_s1, mode='min', factor=0.5, patience=3
)

print(f'Loss: CrossEntropyLoss (label_smoothing={CONFIG["stage1"]["label_smoothing"]})')
print(f'Optimizer: Adam (lr={CONFIG["stage1"]["lr"]}, wd={CONFIG["stage1"]["weight_decay"]})')
print(f'Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)')

In [ ]:
## Training function (shared between Stage 1 and Stage 2)

def train_one_epoch(model, loader, criterion, optimizer, device, extra_loss_fn=None):
    """Train for one epoch.

    Args:
        extra_loss_fn: Optional callable(embeddings) -> loss tensor.
                       Used in Stage 2 for orthogonality penalty.
    """
    model.train()
    total_loss = 0.0
    total_cls_loss = 0.0
    total_extra_loss = 0.0
    correct = 0
    total = 0

    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits, emb = model(images)

        cls_loss = criterion(logits, labels)
        loss = cls_loss

        if extra_loss_fn is not None:
            extra = extra_loss_fn(emb)
            loss = loss + extra
            total_extra_loss += extra.item() * images.size(0)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_cls_loss += cls_loss.item() * images.size(0)
        _, predicted = logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return {
        'loss': total_loss / total,
        'cls_loss': total_cls_loss / total,
        'extra_loss': total_extra_loss / total if extra_loss_fn else 0.0,
        'accuracy': correct / total * 100,
    }


def evaluate(model, loader, criterion, device, extra_loss_fn=None):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0.0
    total_cls_loss = 0.0
    total_extra_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            logits, emb = model(images)

            cls_loss = criterion(logits, labels)
            loss = cls_loss

            if extra_loss_fn is not None:
                extra = extra_loss_fn(emb)
                loss = loss + extra
                total_extra_loss += extra.item() * images.size(0)

            total_loss += loss.item() * images.size(0)
            total_cls_loss += cls_loss.item() * images.size(0)
            _, predicted = logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return {
        'loss': total_loss / total,
        'cls_loss': total_cls_loss / total,
        'extra_loss': total_extra_loss / total if extra_loss_fn else 0.0,
        'accuracy': correct / total * 100,
        'predictions': np.array(all_preds),
        'labels': np.array(all_labels),
    }

print('Training and evaluation functions defined')

In [ ]:
## Quick forward pass sanity check
stage1_model.eval()
with torch.no_grad():
    sample_batch = next(iter(train_loader_s1))
    sample_imgs = sample_batch[0].to(device)
    logits, emb = stage1_model(sample_imgs)

print(f'Input shape:     {sample_imgs.shape}')
print(f'Logits shape:    {logits.shape}')       ## [batch, 7]
print(f'Embedding shape: {emb.shape}')          ## [batch, 256]
print(f'Embedding range: [{emb.min():.4f}, {emb.max():.4f}]')
print(f'Embedding mean:  {emb.mean():.4f}')
print(f'Non-zero dims:   {(emb.abs() > 1e-6).float().mean()*100:.1f}%')

assert logits.shape[1] == 7, f'Expected 7 classes, got {logits.shape[1]}'
assert emb.shape[1] == 256, f'Expected 256-dim embedding, got {emb.shape[1]}'


## Section 5: Stage 1 Training and evaluation


**Expected outcome:** Card identity (7 Pokémon) should be highly separable from slab images. Different Pokémon have different artwork, colours, layouts. The point is to learn a strong identity embedding.

In [ ]:
## Stage 1 Training Loop

s1_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

print(f'STAGE 1: CARD IDENTITY TRAINING')
print(f'Task: {CONFIG["stage1"]["task"]} ({CONFIG["stage1"]["num_classes"]} classes)')
print(f'Epochs: {CONFIG["stage1"]["epochs"]}, Patience: {CONFIG["stage1"]["patience"]}')
print(f'Majority baseline: {train_df["card_name"].value_counts().max()/len(train_df)*100:.1f}%')
print(f'Batch size: {CONFIG["stage1"]["batch_size"]}')

for epoch in range(1, CONFIG['stage1']['epochs'] + 1):
    epoch_start = time.time()

    ## Train
    train_metrics = train_one_epoch(stage1_model, train_loader_s1, criterion_s1,
                                    optimizer_s1, device)
    ## Validate
    val_metrics = evaluate(stage1_model, val_loader_s1, criterion_s1, device)

    ## Scheduler step
    current_lr = optimizer_s1.param_groups[0]['lr']
    scheduler_s1.step(val_metrics['loss'])

    ## Record history
    s1_history['train_loss'].append(train_metrics['loss'])
    s1_history['train_acc'].append(train_metrics['accuracy'])
    s1_history['val_loss'].append(val_metrics['loss'])
    s1_history['val_acc'].append(val_metrics['accuracy'])
    s1_history['lr'].append(current_lr)

    ## Overfitting check
    gap = train_metrics['accuracy'] - val_metrics['accuracy']
    overfit_flag = 'OVERFIT_GAP' if gap > 15 else ''

    elapsed = time.time() - epoch_start
    print(f'Epoch {epoch:2d}/{CONFIG["stage1"]["epochs"]} '
          f'| Train Loss: {train_metrics["loss"]:.4f} Acc: {train_metrics["accuracy"]:5.1f}% '
          f'| Val Loss: {val_metrics["loss"]:.4f} Acc: {val_metrics["accuracy"]:5.1f}% '
          f'| Gap: {gap:+5.1f}% | LR: {current_lr:.1e} '
          f'| {elapsed:.0f}s{overfit_flag}')

    ## Best model checkpoint
    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        best_epoch = epoch
        patience_counter = 0
        torch.save(stage1_model.state_dict(),
                   CONFIG['model_dir'] / 'stage1_identity_best.pth')
    else:
        patience_counter += 1

    ## Early stopping
    if patience_counter >= CONFIG['stage1']['patience']:
        print(f'\nEarly stopping at epoch {epoch} (patience={CONFIG["stage1"]["patience"]})')
        break

print(f'\nBest epoch: {best_epoch}, Best val loss: {best_val_loss:.4f}')
print(f'Model saved: models/vision_v2/stage1_identity_best.pth')

In [ ]:
## Load best model and evaluate on test set
stage1_model.load_state_dict(
    torch.load(CONFIG['model_dir'] / 'stage1_identity_best.pth', map_location=device)
)

test_metrics_s1 = evaluate(stage1_model, test_loader_s1, criterion_s1, device)

## Majority baseline
majority_baseline = test_df['card_name'].value_counts().max() / len(test_df) * 100

print(f'STAGE 1 TEST RESULTS')
print(f'Test Accuracy:     {test_metrics_s1["accuracy"]:.1f}%')
print(f'Majority Baseline: {majority_baseline:.1f}%')
print(f'Lift over baseline: {test_metrics_s1["accuracy"] - majority_baseline:+.1f}pp')
print(f'Test Loss:         {test_metrics_s1["loss"]:.4f}')

In [ ]:
## Classification report and confusion matrix
target_names = [CARD_NAME_INV[i] for i in range(7)]

print('Classification Report:')
print(classification_report(test_metrics_s1['labels'], test_metrics_s1['predictions'],
                            target_names=target_names))

## Confusion matrix
cm = confusion_matrix(test_metrics_s1['labels'], test_metrics_s1['predictions'])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Stage 1: Card identity: Confusion matrix')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'stage1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(s1_history['train_loss']) + 1)

ax1.plot(epochs_range, s1_history['train_loss'], 'b-', label='Train loss')
ax1.plot(epochs_range, s1_history['val_loss'], 'r-', label='Val loss')
ax1.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Stage 1: Loss curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, s1_history['train_acc'], 'b-', label='Train acc')
ax2.plot(epochs_range, s1_history['val_acc'], 'r-', label='Val acc')
ax2.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Stage 1: Accuracy curves')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Stage 1: Card identity encoder training', fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'stage1_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## Save Stage 1 metrics and freeze model
s1_results = {
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'test_accuracy': test_metrics_s1['accuracy'],
    'majority_baseline': majority_baseline,
    'lift': test_metrics_s1['accuracy'] - majority_baseline,
    'test_loss': test_metrics_s1['loss'],
    'history': s1_history,
}

with open(CONFIG['results_dir'] / 'stage1_metrics.json', 'w') as f:
    json.dump(s1_results, f, indent=2, default=str)

## Freeze Stage 1 model entirely for Stage 2
for param in stage1_model.parameters():
    param.requires_grad = False
stage1_model.eval()

## Verify freeze
s1_trainable = sum(p.numel() for p in stage1_model.parameters() if p.requires_grad)
assert s1_trainable == 0, f'Stage 1 still has {s1_trainable} trainable params!'

print(f'Stage 1 metrics saved')
print(f'Stage 1 model frozen: {s1_trainable} trainable params')
print(f'\nConfirmed')
print(f'\nSTAGE 1 COMPLETE')


## Section 6: Stage 2 Condition residual encoder

**Objective:** Train a second encoder to classify grade (PSA 8, 9, 10) from the same slab images, but with an orthogonality constraint that penalises alignment with the Stage 1 identity embedding. This forces the condition embedding into the complementary subspace, capturing physical condition rather than card identity.

**Loss function:** L_total = L_classification + λ · L_orthogonality

Where L_orthogonality = mean(|cosine_similarity(emb_condition, emb_identity)|)

This penalises any correlation between the condition and identity embeddings, pushing them toward orthogonal subspaces. `λ = 0.1 (CONFIG['stage2']['ortho_weight'])`.

**Why cosine similarity and not MSE?** Cosine similarity measures directional alignment regardless of magnitude. Two embeddings can have different scales but still encode the same information if they point in the same direction. Cosine catches this, MSE does not.

In [ ]:
## Stage 2 Model Architecture

class ConditionEncoder(nn.Module):
    """Stage 2: Condition Residual Encoder.

    Shares the same frozen ResNet50 backbone features as Stage 1,
    but has its own embedding head trained with an orthogonality
    constraint against the Stage 1 identity embedding.

    The embedding layer captures physical condition (centering,
    edges, surface, corners) that Stage 1 does not encode.
    """

    def __init__(self, embedding_dim=256, num_classes=3, dropout=0.3):
        super().__init__()

        ## Backbone: ResNet50 pretrained, fully frozen
        ## Separate instance from Stage 1 same weights, independent forward pass
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        ## Freeze entire backbone
        for param in self.features.parameters():
            param.requires_grad = False

        ## Condition embedding head (trainable) separate from Stage 1
        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, embedding_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        ## Grade classifier head (trainable)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            features = self.features(x)
        emb = self.embedding(features)
        logits = self.classifier(emb)
        return logits, emb

    def get_embedding(self, x):
        """Extract condition embedding only."""
        with torch.no_grad():
            features = self.features(x)
            emb = self.embedding(features)
        return emb

print('ConditionEncoder defined')

In [ ]:
## Orthogonality loss function

class OrthogonalityLoss:
    """Penalises alignment between condition and identity embeddings.

    L_ortho = mean(|cosine_similarity(emb_condition, emb_identity)|)

    The absolute value ensures both positive and negative correlation
    are penalised as we want orthogonality, not anti-correlation.
    """

    def __init__(self, identity_model, weight=0.1):
        self.identity_model = identity_model
        self.weight = weight
        self._last_ortho_value = 0.0  ## For logging

    def __call__(self, condition_emb):
        ## Get identity embedding for the same batch
        ## Stage 1 model is frozen and in eval mode. This is a forward pass only
        ## The input images are needed, but there are only condition embeddings here.
        ## Solution: The identity embeddings are stored during the forward pass.
        ## See the modified training loop below.

        ## This will be called with pre-computed identity embeddings
        ## stored in self._current_identity_emb
        identity_emb = self._current_identity_emb

        ## Cosine similarity per sample
        cos_sim = nn.functional.cosine_similarity(condition_emb, identity_emb, dim=1)

        ## Penalise absolute similarity (want orthogonal, not anti-correlated)
        ortho_loss = cos_sim.abs().mean()

        self._last_ortho_value = ortho_loss.item()

        return self.weight * ortho_loss

    def set_identity_embeddings(self, identity_emb):
        """Store identity embeddings for current batch."""
        self._current_identity_emb = identity_emb.detach()

print(f'Orthogonality weight: {CONFIG["stage2"]["ortho_weight"]}')
print('Orthogonality loss defined')

In [ ]:
## Instantiate Stage 2 model and verify parameters
stage2_model = ConditionEncoder(
    embedding_dim=CONFIG['embedding_dim'],
    num_classes=CONFIG['stage2']['num_classes'],
    dropout=CONFIG['dropout']
).to(device)

total_params = sum(p.numel() for p in stage2_model.parameters())
trainable_params = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable_params:>12,}')
print(f'Frozen parameters:    {frozen_params:>12,}')
print(f'Trainable ratio:      {trainable_params/total_params*100:.2f}%')

## Verify backbone is frozen
backbone_trainable = sum(p.numel() for p in stage2_model.features.parameters() if p.requires_grad)
assert backbone_trainable == 0, f'Backbone has {backbone_trainable} trainable params!'
print(f'\nBackbone trainable params: {backbone_trainable} (confirmed frozen)')

In [ ]:
## Create dataloaders for Stage 2 (grade classification)
train_dataset_s2 = CardDataset(train_df, target_col='grade',
                               label_map=GRADE_MAP, transform=train_transform)
val_dataset_s2 = CardDataset(val_df, target_col='grade',
                             label_map=GRADE_MAP, transform=eval_transform)
test_dataset_s2 = CardDataset(test_df, target_col='grade',
                              label_map=GRADE_MAP, transform=eval_transform)

train_loader_s2 = DataLoader(train_dataset_s2, batch_size=CONFIG['stage2']['batch_size'],
                             shuffle=True, num_workers=2, pin_memory=True)
val_loader_s2 = DataLoader(val_dataset_s2, batch_size=CONFIG['stage2']['batch_size'],
                           shuffle=False, num_workers=2, pin_memory=True)
test_loader_s2 = DataLoader(test_dataset_s2, batch_size=CONFIG['stage2']['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)

print(f'Stage 2 dataloaders:')
print(f'Train: {len(train_dataset_s2)} samples, {len(train_loader_s2)} batches')
print(f'Val:   {len(val_dataset_s2)} samples, {len(val_loader_s2)} batches')
print(f'Test:  {len(test_dataset_s2)} samples, {len(test_loader_s2)} batches')

In [ ]:
## Loss, optimizer, scheduler, orthogonality penalty for Stage 2
criterion_s2 = nn.CrossEntropyLoss(
    label_smoothing=CONFIG['stage2']['label_smoothing']  ## 0.1: noisy grade labels
)

optimizer_s2 = optim.Adam(
    filter(lambda p: p.requires_grad, stage2_model.parameters()),
    lr=CONFIG['stage2']['lr'],
    weight_decay=CONFIG['stage2']['weight_decay']
)

scheduler_s2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_s2, mode='min', factor=0.5, patience=3
)

## Orthogonality penalty
ortho_loss_fn = OrthogonalityLoss(
    identity_model=stage1_model,
    weight=CONFIG['stage2']['ortho_weight']
)

print(f'Loss: CrossEntropyLoss (label_smoothing={CONFIG["stage2"]["label_smoothing"]})')
print(f'Optimizer: Adam (lr={CONFIG["stage2"]["lr"]}, wd={CONFIG["stage2"]["weight_decay"]})')
print(f'Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)')
print(f'Orthogonality: λ={CONFIG["stage2"]["ortho_weight"]} (cosine similarity penalty)')

In [ ]:
## Forward pass sanity check to verify that orthogonality computation works
stage2_model.eval()
stage1_model.eval()

with torch.no_grad():
    sample_batch = next(iter(train_loader_s2))
    sample_imgs = sample_batch[0].to(device)

    ## Stage 2 forward
    logits_s2, emb_s2 = stage2_model(sample_imgs)

    ## Stage 1 identity embedding for same images
    _, emb_s1 = stage1_model(sample_imgs)

    ## Compute cosine similarity
    cos_sim = nn.functional.cosine_similarity(emb_s2, emb_s1, dim=1)

print(f'Stage 2 logits shape:    {logits_s2.shape}')     ## [batch, 3]
print(f'Stage 2 embedding shape: {emb_s2.shape}')        ## [batch, 256]
print(f'Stage 1 embedding shape: {emb_s1.shape}')        ## [batch, 256]
print(f'\nCosine similarity (before training):')
print(f'Mean: {cos_sim.mean():.4f}')
print(f'Std:  {cos_sim.std():.4f}')
print(f'Range: [{cos_sim.min():.4f}, {cos_sim.max():.4f}]')
print(f'Mean |cos_sim|: {cos_sim.abs().mean():.4f}')
print(f'\nThis is the initial alignment before orthogonality training.')
print(f'It should decrease during training as the penalty takes effect.')

assert logits_s2.shape[1] == 3, f'Expected 3 classes, got {logits_s2.shape[1]}'
assert emb_s2.shape[1] == 256, f'Expected 256-dim, got {emb_s2.shape[1]}'
print(f'\nStage 2 forward pass and orthogonality computation verified')


## Section 7: Stage 2 Training and evaluation

**Expected outcome:** Grade classification accuracy will be lower than card identity (Stage 1 got 91.3%). PSA 8–10 condition differences are subtle. We expect 45-60% accuracy (v1 achieved 52.1% without orthogonality). The key question is not raw accuracy but whether the orthogonality constraint produces embeddings that better separate condition from identity.

**Training loop modification:** Unlike Stage 1, each batch requires two forward passes:
1. Stage 1 (frozen) to get identity embeddings
2. Stage 2 (training) to get condition embeddings + grade logits

The orthogonality loss is computed between the two embeddings.

In [ ]:
## Stage 2 Training Loop: custom loop with orthogonality penalty

def train_one_epoch_s2(stage2_model, stage1_model, loader, criterion, optimizer,
                        ortho_loss_fn, device):
    """Train Stage 2 for one epoch with orthogonality constraint."""
    stage2_model.train()
    stage1_model.eval()  ## Always frozen

    total_loss = 0.0
    total_cls_loss = 0.0
    total_ortho_loss = 0.0
    correct = 0
    total = 0
    cos_sims = []  ## Track cosine similarity over epoch

    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        ## Forward pass: Stage 2
        logits, cond_emb = stage2_model(images)

        ## Forward pass: Stage 1 (frozen, no grad)
        with torch.no_grad():
            _, identity_emb = stage1_model(images)

        ## Classification loss
        cls_loss = criterion(logits, labels)

        ## Orthogonality loss
        cos_sim = nn.functional.cosine_similarity(cond_emb, identity_emb, dim=1)
        ortho_loss = cos_sim.abs().mean()
        weighted_ortho = ortho_loss_fn.weight * ortho_loss

        ## Total loss
        loss = cls_loss + weighted_ortho

        loss.backward()
        optimizer.step()

        ## Track metrics
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_cls_loss += cls_loss.item() * batch_size
        total_ortho_loss += weighted_ortho.item() * batch_size
        _, predicted = logits.max(1)
        correct += predicted.eq(labels).sum().item()
        total += batch_size
        cos_sims.append(cos_sim.detach().cpu())

    all_cos = torch.cat(cos_sims)
    return {
        'loss': total_loss / total,
        'cls_loss': total_cls_loss / total,
        'ortho_loss': total_ortho_loss / total,
        'accuracy': correct / total * 100,
        'mean_cos_sim': all_cos.abs().mean().item(),
    }


def evaluate_s2(stage2_model, stage1_model, loader, criterion, ortho_loss_fn, device):
    """Evaluate Stage 2 with orthogonality metrics."""
    stage2_model.eval()
    stage1_model.eval()

    total_loss = 0.0
    total_cls_loss = 0.0
    total_ortho_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    cos_sims = []

    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)

            logits, cond_emb = stage2_model(images)
            _, identity_emb = stage1_model(images)

            cls_loss = criterion(logits, labels)
            cos_sim = nn.functional.cosine_similarity(cond_emb, identity_emb, dim=1)
            ortho_loss = cos_sim.abs().mean()
            weighted_ortho = ortho_loss_fn.weight * ortho_loss
            loss = cls_loss + weighted_ortho

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_cls_loss += cls_loss.item() * batch_size
            total_ortho_loss += weighted_ortho.item() * batch_size
            _, predicted = logits.max(1)
            correct += predicted.eq(labels).sum().item()
            total += batch_size
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            cos_sims.append(cos_sim.cpu())

    all_cos = torch.cat(cos_sims)
    return {
        'loss': total_loss / total,
        'cls_loss': total_cls_loss / total,
        'ortho_loss': total_ortho_loss / total,
        'accuracy': correct / total * 100,
        'mean_cos_sim': all_cos.abs().mean().item(),
        'predictions': np.array(all_preds),
        'labels': np.array(all_labels),
    }

print('Stage 2 training and evaluation functions defined')

In [ ]:
## Stage 2 Training

s2_history = {
    'train_loss': [], 'train_cls_loss': [], 'train_ortho_loss': [],
    'train_acc': [], 'train_cos_sim': [],
    'val_loss': [], 'val_cls_loss': [], 'val_ortho_loss': [],
    'val_acc': [], 'val_cos_sim': [],
    'lr': []
}

best_val_loss_s2 = float('inf')
patience_counter_s2 = 0
best_epoch_s2 = 0

## Majority baseline for grade
majority_grade = train_df['grade'].value_counts().max() / len(train_df) * 100

print(f'STAGE 2: CONDITION RESIDUAL TRAINING')
print(f'Task: {CONFIG["stage2"]["task"]} ({CONFIG["stage2"]["num_classes"]} classes)')
print(f'Epochs: {CONFIG["stage2"]["epochs"]}, Patience: {CONFIG["stage2"]["patience"]}')
print(f'Orthogonality weight: λ={CONFIG["stage2"]["ortho_weight"]}')
print(f'Majority baseline: {majority_grade:.1f}%')
print(f'Batch size: {CONFIG["stage2"]["batch_size"]}')
print(f'Learning rate: {CONFIG["stage2"]["lr"]}')
print(f'Weight decay: {CONFIG["stage2"]["weight_decay"]}')
print(f'Label smoothing: {CONFIG["stage2"]["label_smoothing"]}')

for epoch in range(1, CONFIG['stage2']['epochs'] + 1):
    epoch_start = time.time()

    ## Train
    train_m = train_one_epoch_s2(stage2_model, stage1_model, train_loader_s2,
                                 criterion_s2, optimizer_s2, ortho_loss_fn, device)
    ## Validate
    val_m = evaluate_s2(stage2_model, stage1_model, val_loader_s2,
                        criterion_s2, ortho_loss_fn, device)

    ## Scheduler
    current_lr = optimizer_s2.param_groups[0]['lr']
    scheduler_s2.step(val_m['loss'])

    ## Record history
    for key in ['loss', 'cls_loss', 'ortho_loss', 'accuracy', 'cos_sim']:
        train_key = f'train_{key}' if key != 'accuracy' else 'train_acc'
        val_key = f'val_{key}' if key != 'accuracy' else 'val_acc'
        if key == 'accuracy':
            s2_history['train_acc'].append(train_m['accuracy'])
            s2_history['val_acc'].append(val_m['accuracy'])
        elif key == 'cos_sim':
            s2_history['train_cos_sim'].append(train_m['mean_cos_sim'])
            s2_history['val_cos_sim'].append(val_m['mean_cos_sim'])
        else:
            s2_history[f'train_{key}'].append(train_m[key])
            s2_history[f'val_{key}'].append(val_m[key])
    s2_history['lr'].append(current_lr)

    ## Overfitting check
    gap = train_m['accuracy'] - val_m['accuracy']
    overfit_flag = 'OVERFIT' if gap > 15 else ''

    elapsed = time.time() - epoch_start
    print(f'Epoch {epoch:2d}/{CONFIG["stage2"]["epochs"]} '
          f'| Loss: {train_m["loss"]:.4f}/{val_m["loss"]:.4f} '
          f'| ClsL: {train_m["cls_loss"]:.4f}/{val_m["cls_loss"]:.4f} '
          f'| OrtL: {train_m["ortho_loss"]:.4f}/{val_m["ortho_loss"]:.4f} '
          f'| Acc: {train_m["accuracy"]:5.1f}/{val_m["accuracy"]:5.1f}% '
          f'| |cos|: {train_m["mean_cos_sim"]:.3f}/{val_m["mean_cos_sim"]:.3f} '
          f'| {elapsed:.0f}s{overfit_flag}')

    # Best model checkpoint
    if val_m['loss'] < best_val_loss_s2:
        best_val_loss_s2 = val_m['loss']
        best_epoch_s2 = epoch
        patience_counter_s2 = 0
        torch.save(stage2_model.state_dict(),
                   CONFIG['model_dir'] / 'stage2_condition_best.pth')
    else:
        patience_counter_s2 += 1

    # Early stopping
    if patience_counter_s2 >= CONFIG['stage2']['patience']:
        print(f'\nEarly stopping at epoch {epoch} (patience={CONFIG["stage2"]["patience"]})')
        break

print(f'\nBest epoch: {best_epoch_s2}, Best val loss: {best_val_loss_s2:.4f}')
print(f'Model saved: models/vision_v2/stage2_condition_best.pth')

In [ ]:
## Load best Stage 2 model and evaluate on test set
stage2_model.load_state_dict(
    torch.load(CONFIG['model_dir'] / 'stage2_condition_best.pth', map_location=device)
)

test_metrics_s2 = evaluate_s2(stage2_model, stage1_model, test_loader_s2,
                               criterion_s2, ortho_loss_fn, device)

## Baselines
majority_grade_test = test_df['grade'].value_counts().max() / len(test_df) * 100

print(f'STAGE 2 TEST RESULTS')
print(f'Test Accuracy:      {test_metrics_s2["accuracy"]:.1f}%')
print(f'Majority Baseline:  {majority_grade_test:.1f}%')
print(f'Lift over baseline: {test_metrics_s2["accuracy"] - majority_grade_test:+.1f}pp')
print(f'Test Loss (total):  {test_metrics_s2["loss"]:.4f}')
print(f'Test Loss (cls):    {test_metrics_s2["cls_loss"]:.4f}')
print(f'Test Loss (ortho):  {test_metrics_s2["ortho_loss"]:.4f}')
print(f'Test |cos_sim|:     {test_metrics_s2["mean_cos_sim"]:.4f}')
print(f'\nv1 reference: 52.1% accuracy (no orthogonality, 1123 samples)')

In [ ]:
## Classification report, confusion matrix, and ordinal error analysis
grade_names = ['PSA 8', 'PSA 9', 'PSA 10']

print('Classification report:')
print(classification_report(test_metrics_s2['labels'], test_metrics_s2['predictions'],
                            target_names=grade_names))

## Confusion matrix
cm_s2 = confusion_matrix(test_metrics_s2['labels'], test_metrics_s2['predictions'])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_s2, annot=True, fmt='d', cmap='Blues',
            xticklabels=grade_names, yticklabels=grade_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Stage 2 Grade Classification: Confusion matrix')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'stage2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Ordinal error analysis
preds_grades = np.array([GRADE_INV[p] for p in test_metrics_s2['predictions']])
true_grades = np.array([GRADE_INV[l] for l in test_metrics_s2['labels']])
errors = np.abs(preds_grades - true_grades)

print(f'\nOrdinal error analysis:')
print(f'Exact match (error=0): {(errors == 0).sum()} ({(errors == 0).mean()*100:.1f}%)')
print(f'Adjacent error (±1):   {(errors == 1).sum()} ({(errors == 1).mean()*100:.1f}%)')
print(f'Far error (±2):        {(errors == 2).sum()} ({(errors == 2).mean()*100:.1f}%)')
print(f'Grade MAE:             {errors.mean():.3f}')
print(f'Adjacent error ratio:  {(errors == 1).sum() / max((errors > 0).sum(), 1)*100:.1f}% of all errors')

In [ ]:
## Training curves. 4-panel: loss, accuracy, orthogonality, cosine similarity
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(s2_history['train_loss']) + 1)

## Loss
axes[0, 0].plot(epochs_range, s2_history['train_loss'], 'b-', label='Train total')
axes[0, 0].plot(epochs_range, s2_history['val_loss'], 'r-', label='Val total')
axes[0, 0].axvline(x=best_epoch_s2, color='g', linestyle='--', alpha=0.5, label=f'Best ({best_epoch_s2})')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Total loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

## Accuracy
axes[0, 1].plot(epochs_range, s2_history['train_acc'], 'b-', label='Train')
axes[0, 1].plot(epochs_range, s2_history['val_acc'], 'r-', label='Val')
axes[0, 1].axhline(y=majority_grade, color='gray', linestyle=':', label=f'Majority ({majority_grade:.1f}%)')
axes[0, 1].axvline(x=best_epoch_s2, color='g', linestyle='--', alpha=0.5)
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].set_title('Grade accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

## Classification vs Orthogonality loss
axes[1, 0].plot(epochs_range, s2_history['train_cls_loss'], 'b-', label='Train Cls')
axes[1, 0].plot(epochs_range, s2_history['val_cls_loss'], 'r-', label='Val Cls')
axes[1, 0].plot(epochs_range, s2_history['train_ortho_loss'], 'b--', alpha=0.6, label='Train Ortho')
axes[1, 0].plot(epochs_range, s2_history['val_ortho_loss'], 'r--', alpha=0.6, label='Val Ortho')
axes[1, 0].axvline(x=best_epoch_s2, color='g', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Classification vs Orthogonality loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Cosine similarity
axes[1, 1].plot(epochs_range, s2_history['train_cos_sim'], 'b-', label='Train |cos_sim|')
axes[1, 1].plot(epochs_range, s2_history['val_cos_sim'], 'r-', label='Val |cos_sim|')
axes[1, 1].axvline(x=best_epoch_s2, color='g', linestyle='--', alpha=0.5)
axes[1, 1].axhline(y=0, color='gray', linestyle=':', alpha=0.5)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Mean |Cosine similarity|')
axes[1, 1].set_title('Identity-Condition alignment (lower = more orthogonal)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Stage 2: Condition residual encoder training', fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'stage2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## Save Stage 2 metrics
s2_results = {
    'best_epoch': best_epoch_s2,
    'best_val_loss': best_val_loss_s2,
    'test_accuracy': test_metrics_s2['accuracy'],
    'majority_baseline': majority_grade_test,
    'lift': test_metrics_s2['accuracy'] - majority_grade_test,
    'test_loss_total': test_metrics_s2['loss'],
    'test_loss_cls': test_metrics_s2['cls_loss'],
    'test_loss_ortho': test_metrics_s2['ortho_loss'],
    'test_cos_sim': test_metrics_s2['mean_cos_sim'],
    'grade_mae': float(errors.mean()),
    'adjacent_error_ratio': float((errors == 1).sum() / max((errors > 0).sum(), 1)),
    'v1_reference_accuracy': 52.1,
    'history': s2_history,
}

with open(CONFIG['results_dir'] / 'stage2_metrics.json', 'w') as f:
    json.dump(s2_results, f, indent=2, default=str)

print(f'Stage 2 metrics saved')
print(f'\nSTAGE 2 COMPLETE')


## Section 8: Embedding extraction

**Objective:** Extract 256-dim condition embeddings from Stage 2 for all 3,812 samples. These are the intrinsic representations that flow to the fusion module. We also extract Stage 1 identity embeddings for comparison and diagnostic purposes.

In [ ]:
## Create full-dataset loader (no augmentation, deterministic order)
full_dataset = CardDataset(df_all, target_col='grade',
                           label_map=GRADE_MAP, transform=eval_transform)
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Full dataset: {len(full_dataset)} samples')
print(f'Batches: {len(full_loader)}')

In [ ]:
## Extract embeddings from both stages
stage1_model.eval()
stage2_model.eval()

all_condition_embs = []
all_identity_embs = []
all_indices = []

with torch.no_grad():
    for images, labels, indices in full_loader:
        images = images.to(device)

        ## Stage 2: condition embeddings (primary output for fusion)
        _, cond_emb = stage2_model(images)
        all_condition_embs.append(cond_emb.cpu().numpy())

        ## Stage 1: identity embeddings (diagnostic)
        _, id_emb = stage1_model(images)
        all_identity_embs.append(id_emb.cpu().numpy())

        all_indices.extend(indices.numpy())

condition_embs = np.vstack(all_condition_embs)
identity_embs = np.vstack(all_identity_embs)
indices = np.array(all_indices)

print(f'Condition embeddings: {condition_embs.shape}')
print(f'Identity embeddings:  {identity_embs.shape}')
print(f'Indices:              {indices.shape}')

In [ ]:
## Build embedding dataframe with metadata
## Map indices back to df_all rows
emb_df = df_all.iloc[indices].reset_index(drop=True).copy()

## Add condition embedding columns
cond_cols = [f'cond_emb_{i}' for i in range(256)]
cond_emb_df = pd.DataFrame(condition_embs, columns=cond_cols)
emb_df = pd.concat([emb_df[['listing_id', 'card_name', 'grade', 'price', 'split']], cond_emb_df], axis=1)

## Verify integrity
assert len(emb_df) == len(df_all), f'Row count mismatch: {len(emb_df)} vs {len(df_all)}'
assert emb_df['listing_id'].nunique() == len(emb_df), 'Duplicate listing_ids in embeddings!'

print(f'Embedding dataframe: {emb_df.shape}')
print(f'Columns: listing_id, card_name, grade, price, split, cond_emb_0...cond_emb_255')
print(f'\nSplit distribution:')
print(emb_df['split'].value_counts().to_string())

## Check for NaN or zero vectors
nan_count = emb_df[cond_cols].isna().any(axis=1).sum()
zero_count = (emb_df[cond_cols].abs().sum(axis=1) < 1e-8).sum()
print(f'\nNaN vectors: {nan_count}')
print(f'Zero vectors: {zero_count}')

assert nan_count == 0, f'{nan_count} NaN vectors!'

In [ ]:
## Also build identity embedding dataframe for diagnostics
id_cols = [f'id_emb_{i}' for i in range(256)]
id_emb_df = pd.DataFrame(identity_embs, columns=id_cols)
id_emb_full = pd.concat([
    df_all.iloc[indices].reset_index(drop=True)[['listing_id', 'card_name', 'grade', 'split']],
    id_emb_df
], axis=1)

print(f'Identity embedding dataframe: {id_emb_full.shape}')

In [ ]:
## Save condition embeddings (primary output for fusion module)
emb_save_path = CONFIG['embeddings_dir'] / 'visual_embeddings_v2.parquet'
emb_df.to_parquet(emb_save_path, index=False)

## Also save identity embeddings for diagnostic reference
id_save_path = CONFIG['embeddings_dir'] / 'identity_embeddings_v2.parquet'
id_emb_full.to_parquet(id_save_path, index=False)

## Verify saved file
verify = pd.read_parquet(emb_save_path)
print(f'Saved condition embeddings: {verify.shape}')
print(f'Saved identity embeddings:  {id_emb_full.shape}')
print(f'\nCondition file: {emb_save_path}')
print(f'Identity file:  {id_save_path}')

assert verify.shape == emb_df.shape, 'Save verification failed!'


## Section 9: Representation quality assessment

**Objective:** Evaluate the quality of the Stage 2 condition embeddings. This section directly addresses condition-encoder's representational criterion: does the embedding capture grading-relevant structure?

We evaluate:
1. Global grade separability (silhouette score)
2. Local grade structure (nearest-neighbour lift)
3. Within-card ordinal consistency
4. Dimensionality and effective rank
5. Visualisation (t-SNE/UMAP)
6. Comparison against Stage 1 identity embeddings

In [ ]:
## Global grade separability: silhouette score
## Use test set only for unbiased evaluation
test_mask = emb_df['split'] == 'test'
test_embs = emb_df.loc[test_mask, cond_cols].values
test_grades = emb_df.loc[test_mask, 'grade'].values

sil_condition = silhouette_score(test_embs, test_grades)

## Also compute for identity embeddings
test_id_embs = id_emb_full.loc[id_emb_full['split'] == 'test', id_cols].values
test_id_grades = id_emb_full.loc[id_emb_full['split'] == 'test', 'grade'].values
sil_identity_grade = silhouette_score(test_id_embs, test_id_grades)

## Identity embeddings by card_name (should be high)
test_id_names = id_emb_full.loc[id_emb_full['split'] == 'test', 'card_name'].values
sil_identity_name = silhouette_score(test_id_embs, test_id_names)

print('SILHOUETTE SCORES')
print(f'Condition embeddings by grade:   {sil_condition:.4f}')
print(f'Identity embeddings by grade:    {sil_identity_grade:.4f}')
print(f'Identity embeddings by card_name:{sil_identity_name:.4f}')
print(f'\nv1 reference (condition by grade): 0.033')
print(f'\nInterpretation:')
print(f'Identity by card_name should be high (identity is well-separated)')
print(f'Identity by grade should be low (identity does not encode grade)')
print(f'Condition by grade: higher than identity-by-grade = condition carries grade signal')

In [ ]:
## Nearest-neighbour grade lift

def compute_nn_lift(embeddings, labels, k=5):
    """Compute how much more likely nearest neighbours share the same grade
    compared to random chance."""
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine')
    nn.fit(embeddings)
    distances, indices_nn = nn.kneighbors(embeddings)

    ## Exclude self (index 0)
    neighbor_indices = indices_nn[:, 1:]

    same_grade_count = 0
    total_neighbors = 0

    for i in range(len(labels)):
        for j in neighbor_indices[i]:
            if labels[i] == labels[j]:
                same_grade_count += 1
            total_neighbors += 1

    actual_rate = same_grade_count / total_neighbors

    ## Expected rate under random assignment
    grade_counts = Counter(labels)
    n = len(labels)
    expected_rate = sum((c/n)**2 for c in grade_counts.values())

    lift = actual_rate / expected_rate
    return actual_rate, expected_rate, lift

## Condition embeddings
cond_actual, cond_expected, cond_lift = compute_nn_lift(test_embs, test_grades, k=5)

## Identity embeddings (by grade - should show NO lift)
id_actual, id_expected, id_lift = compute_nn_lift(test_id_embs, test_id_grades, k=5)

print('NEAREST-NEIGHBOUR GRADE LIFT (k=5)')
print(f'Condition embeddings:')
print(f'Same-grade rate: {cond_actual:.4f} (expected: {cond_expected:.4f})')
print(f'Lift: {cond_lift:.3f}x')
print(f'\nIdentity embeddings:')
print(f'Same-grade rate: {id_actual:.4f} (expected: {id_expected:.4f})')
print(f'Lift: {id_lift:.3f}x')
print(f'\nCondition lift > Identity lift = condition captures grade structure that identity does not')

In [ ]:
## Within-card ordinal consistency
## For each card_name, check if embedding distances follow grade ordering:
## dist(PSA8, PSA10) > dist(PSA8, PSA9) > dist(PSA9, PSA10)

test_cond_df = emb_df[test_mask].copy()

ordinal_results = []

for card_name in sorted(test_cond_df['card_name'].unique()):
    card_data = test_cond_df[test_cond_df['card_name'] == card_name]

    ## Need at least 2 samples per grade
    grade_counts = card_data['grade'].value_counts()
    if not all(g in grade_counts.index and grade_counts[g] >= 2 for g in [8, 9, 10]):
        continue

    ## Mean embedding per grade
    mean_embs = {}
    for g in [8, 9, 10]:
        mean_embs[g] = card_data.loc[card_data['grade'] == g, cond_cols].values.mean(axis=0)

    ## Pairwise distances
    d_8_9 = np.linalg.norm(mean_embs[8] - mean_embs[9])
    d_9_10 = np.linalg.norm(mean_embs[9] - mean_embs[10])
    d_8_10 = np.linalg.norm(mean_embs[8] - mean_embs[10])

    ## Ordinal check: d(8,10) > d(8,9) AND d(8,10) > d(9,10)
    ordinal_ok = d_8_10 > d_8_9 and d_8_10 > d_9_10

    ordinal_results.append({
        'card_name': card_name,
        'd_8_9': d_8_9,
        'd_9_10': d_9_10,
        'd_8_10': d_8_10,
        'ordinal_consistent': ordinal_ok,
        'n_per_grade': {g: int(grade_counts.get(g, 0)) for g in [8, 9, 10]}
    })

ord_df = pd.DataFrame(ordinal_results)
print('WITHIN-CARD ORDINAL CONSISTENCY')
print(f'Cards tested: {len(ord_df)}')
print(f'Ordinal consistent: {ord_df["ordinal_consistent"].sum()}/{len(ord_df)} '
      f'({ord_df["ordinal_consistent"].mean()*100:.1f}%)')
print(f'\nPer-card results:')
for _, row in ord_df.iterrows():
    mark = 'YES' if row['ordinal_consistent'] else 'NO'
    print(f'{mark} {row["card_name"]:12s}: d(8,9)={row["d_8_9"]:.3f}  '
          f'd(9,10)={row["d_9_10"]:.3f}  d(8,10)={row["d_8_10"]:.3f}')

In [ ]:
## Effective dimensionality
## Condition embeddings
pca_cond = PCA(n_components=min(256, len(test_embs)))
pca_cond.fit(test_embs)
cumvar_cond = np.cumsum(pca_cond.explained_variance_ratio_)

## Effective dims at 90% and 95% variance
eff_90_cond = np.searchsorted(cumvar_cond, 0.90) + 1
eff_95_cond = np.searchsorted(cumvar_cond, 0.95) + 1

## Identity embeddings
pca_id = PCA(n_components=min(256, len(test_id_embs)))
pca_id.fit(test_id_embs)
cumvar_id = np.cumsum(pca_id.explained_variance_ratio_)

eff_90_id = np.searchsorted(cumvar_id, 0.90) + 1
eff_95_id = np.searchsorted(cumvar_id, 0.95) + 1

print('EFFECTIVE DIMENSIONALITY')
print(f'Condition embeddings:')
print(f'90% variance: {eff_90_cond} dims')
print(f'95% variance: {eff_95_cond} dims')
print(f'Top-1 PC explains: {pca_cond.explained_variance_ratio_[0]*100:.1f}%')
print(f'\nIdentity embeddings:')
print(f'90% variance: {eff_90_id} dims')
print(f'95% variance: {eff_95_id} dims')
print(f'Top-1 PC explains: {pca_id.explained_variance_ratio_[0]*100:.1f}%')

## Plot cumulative variance
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(cumvar_cond)+1), cumvar_cond, 'b-', label='Condition')
ax.plot(range(1, len(cumvar_id)+1), cumvar_id, 'r-', label='Identity')
ax.axhline(y=0.90, color='gray', linestyle=':', alpha=0.5, label='90%')
ax.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5, label='95%')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative variance explained')
ax.set_title('PCA: Condition vs Identity embeddings')
ax.set_xlim(0, 100)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'pca_variance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## t-SNE visualisation: condition embeddings coloured by grade and by card_name

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
tsne_cond = tsne.fit_transform(test_embs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

## By grade
grade_colors = {8: '#e74c3c', 9: '#f39c12', 10: '#27ae60'}
for g in [8, 9, 10]:
    mask = test_grades == g
    ax1.scatter(tsne_cond[mask, 0], tsne_cond[mask, 1],
               c=grade_colors[g], alpha=0.5, s=15, label=f'PSA {g}')
ax1.set_title('Condition Embeddings by Grade')
ax1.legend()
ax1.set_xticks([])
ax1.set_yticks([])

## By card_name
test_names = test_cond_df['card_name'].values
name_colors = plt.cm.Set1(np.linspace(0, 1, 7))
for i, name in enumerate(sorted(test_cond_df['card_name'].unique())):
    mask = test_names == name
    ax2.scatter(tsne_cond[mask, 0], tsne_cond[mask, 1],
               c=[name_colors[i]], alpha=0.5, s=15, label=name)
ax2.set_title('Condition Embeddings by Card Name')
ax2.legend(fontsize=8)
ax2.set_xticks([])
ax2.set_yticks([])

plt.suptitle('Stage 2: Condition Embedding Structure (t-SNE)', fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'tsne_condition_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nIf orthogonality worked: card_name clusters should be less distinct here than in identity space')

In [ ]:
## t-SNE for identity embeddings. It should show clear card_name clusters
tsne_id = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
tsne_id_coords = tsne_id.fit_transform(test_id_embs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

## By card_name
for i, name in enumerate(sorted(test_cond_df['card_name'].unique())):
    mask = test_names == name
    ax1.scatter(tsne_id_coords[mask, 0], tsne_id_coords[mask, 1],
               c=[name_colors[i]], alpha=0.5, s=15, label=name)
ax1.set_title('Identity Embeddings by Card Name')
ax1.legend(fontsize=8)
ax1.set_xticks([])
ax1.set_yticks([])

## By grade
for g in [8, 9, 10]:
    mask = test_id_grades == g
    ax2.scatter(tsne_id_coords[mask, 0], tsne_id_coords[mask, 1],
               c=grade_colors[g], alpha=0.5, s=15, label=f'PSA {g}')
ax2.set_title('Identity Embeddings by Grade')
ax2.legend()
ax2.set_xticks([])
ax2.set_yticks([])

plt.suptitle('Stage 1: Identity Embedding Structure (t-SNE)', fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'tsne_identity_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Identity embeddings should show clear card_name clusters but no grade structure')

In [ ]:
## Cross-embedding orthogonality verification on full test set
cos_sims_full = nn.functional.cosine_similarity(
    torch.tensor(test_embs, dtype=torch.float32),
    torch.tensor(test_id_embs, dtype=torch.float32),
    dim=1
).numpy()

print('ORTHOGONALITY VERIFICATION (Test Set)')
print(f'Mean cosine similarity:     {cos_sims_full.mean():.6f}')
print(f'Mean |cosine similarity|:   {np.abs(cos_sims_full).mean():.6f}')
print(f'Std:                        {cos_sims_full.std():.6f}')
print(f'Range:                      [{cos_sims_full.min():.6f}, {cos_sims_full.max():.6f}]')
print(f'\n|cos_sim| < 0.01 for {(np.abs(cos_sims_full) < 0.01).mean()*100:.1f}% of samples')
print(f'|cos_sim| < 0.05 for {(np.abs(cos_sims_full) < 0.05).mean()*100:.1f}% of samples')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cos_sims_full, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', label='Perfect orthogonality')
ax.set_xlabel('Cosine Similarity (Condition vs Identity)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Condition-Identity Cosine Similarity')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'orthogonality_histogram.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 10: Economic relevance (Embedding-price correlation)

**Objective:** To test whether condition embeddings carry price-relevant information. Price is not used in training, this is a post-hoc evaluation. If condition embeddings correlate with price after controlling for card identity (which is orthogonal), it suggests the condition signal has economic meaning.

In [ ]:
## PCA of condition embeddings. Correlate components with price

test_cond_with_price = emb_df[emb_df['split'] == 'test'].copy()
test_cond_vals = test_cond_with_price[cond_cols].values
test_prices = test_cond_with_price['price'].values

## PCA on test condition embeddings
pca_price = PCA(n_components=10)
pca_scores = pca_price.fit_transform(test_cond_vals)

print('CONDITION EMBEDDING PCA-PRICE CORRELATIONS (Spearman)')
sig_count = 0
for i in range(10):
    rho, pval = spearmanr(pca_scores[:, i], test_prices)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    if pval < 0.05:
        sig_count += 1
    print(f'PC{i+1:2d} (var={pca_price.explained_variance_ratio_[i]*100:5.1f}%): '
          f'ρ={rho:+.4f}  p={pval:.4f} {sig}')

print(f'\nSignificant components (p<0.05): {sig_count}/10')

In [ ]:
## Same analysis for identity embeddings for comparison
test_id_vals = id_emb_full[id_emb_full['split'] == 'test'][id_cols].values

pca_id_price = PCA(n_components=10)
pca_id_scores = pca_id_price.fit_transform(test_id_vals)

print('IDENTITY EMBEDDING PCA-PRICE CORRELATIONS (Spearman)')
sig_count_id = 0
for i in range(10):
    rho, pval = spearmanr(pca_id_scores[:, i], test_prices)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    if pval < 0.05:
        sig_count_id += 1
    print(f'  PC{i+1:2d} (var={pca_id_price.explained_variance_ratio_[i]*100:5.1f}%): '
          f'ρ={rho:+.4f}  p={pval:.4f} {sig}')

print(f'\nSignificant components (p<0.05): {sig_count_id}/10')
print(f'\nInterpretation:')
print(f'Identity-price correlation reflects card-level price differences (Charizard > Venusaur)')
print(f'Condition-price correlation reflects within-card condition premium (PSA 10 > PSA 8)')
print(f'Both are expected. Condition correlation is more interesting for condition-encoder.')

**Interpretation:** Identity-price correlation reflects card-level price differences `Charizard > Venusaur'`. Condition-price correlation reflects within-card condition premium `PSA 10 > PSA 8`
Both are expected. Condition correlation is more interesting for `condition-encoder.`

In [ ]:
## Within-card price correlation. Does condition embedding predict price
## within the same Pokémon? This controls for card identity.

print('WITHIN-CARD CONDITION-PRICE CORRELATION')
within_card_results = []

for card_name in sorted(test_cond_with_price['card_name'].unique()):
    card_mask = test_cond_with_price['card_name'] == card_name
    card_embs = test_cond_with_price.loc[card_mask, cond_cols].values
    card_prices = test_cond_with_price.loc[card_mask, 'price'].values

    if len(card_prices) < 10:
        continue

    ## Use first 3 PCs of condition embedding
    pca_card = PCA(n_components=3)
    card_pca = pca_card.fit_transform(card_embs)

    best_rho = 0
    best_p = 1.0
    best_pc = 0
    for pc in range(3):
        rho, pval = spearmanr(card_pca[:, pc], card_prices)
        if abs(rho) > abs(best_rho):
            best_rho = rho
            best_p = pval
            best_pc = pc + 1

    sig = '*' if best_p < 0.05 else ''
    within_card_results.append({
        'card_name': card_name, 'n': int(card_mask.sum()),
        'best_rho': best_rho, 'best_p': best_p, 'best_pc': best_pc
    })
    print(f'{card_name:12s} (n={card_mask.sum():3d}): '
          f'best ρ={best_rho:+.4f} (PC{best_pc}) p={best_p:.4f} {sig}')

sig_within = sum(1 for r in within_card_results if r['best_p'] < 0.05)
print(f'\nSignificant within-card correlations: {sig_within}/{len(within_card_results)}')

In [ ]:
## Log-price correlation (price range is $1-$38K, log may be more appropriate)
test_log_prices = np.log1p(test_prices)

print('CONDITION PC vs LOG-PRICE CORRELATIONS')
for i in range(5):
    rho, pval = spearmanr(pca_scores[:, i], test_log_prices)
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f'PC{i+1}: ρ={rho:+.4f}  p={pval:.4f} {sig}')


## Section 11: Shortcut detection

**Objective:** Verify the condition encoder attends to card body regions (edges, corners, surface) rather than PSA label text, slab casing, or background. This is the interpretive criterion of `condition-encoder`.

In [ ]:
## Override Stage 2 forward to allow grad flow for Grad-CAM

stage2_model._original_forward = stage2_model.forward

def gradcam_forward(self, x):
    features = self.features(x)
    emb = self.embedding(features)
    logits = self.classifier(emb)
    return logits, emb

stage2_model.forward = lambda x: gradcam_forward(stage2_model, x)
print('Forward override applied for Grad-CAM')

In [ ]:
## Grad-CAM implementation for Stage 2
class GradCAM:
    """Grad-CAM for ResNet50-based encoder."""

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        # Temporarily enable grad for entire model
        prev_requires_grad = {}
        for name, param in self.model.named_parameters():
            prev_requires_grad[name] = param.requires_grad
            param.requires_grad = True

        self.model.eval()
        input_tensor.requires_grad = True

        output, _ = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot)

        # Restore original requires_grad
        for name, param in self.model.named_parameters():
            param.requires_grad = prev_requires_grad[name]

        if self.gradients is None:
            return np.zeros((7, 7))  # Fallback

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam.squeeze().cpu().numpy()

# Target: last conv layer of ResNet50
target_layer = stage2_model.features[7][-1]
gradcam = GradCAM(stage2_model, target_layer)

print('Grad-CAM initialised (fixed for frozen backbone)')

In [ ]:
## Generate Grad-CAM visualisations. 3 samples per grade
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row, grade in enumerate([8, 9, 10]):
    samples = test_df[test_df['grade'] == grade].sample(3, random_state=SEED)

    for col, (_, sample) in enumerate(samples.iterrows()):
        img_path = PROJECT_ROOT / sample['local_image_path']
        raw_img = Image.open(img_path).convert('RGB')
        input_tensor = eval_transform(raw_img).unsqueeze(0).to(device)

        cam = gradcam.generate(input_tensor)

        ## Resize CAM to image size
        cam_resized = np.array(Image.fromarray(cam).resize((224, 224), Image.BILINEAR))

        ## Display
        raw_resized = raw_img.resize((224, 224))
        axes[row, col].imshow(raw_resized)
        axes[row, col].imshow(cam_resized, cmap='jet', alpha=0.4)
        axes[row, col].set_title(f"PSA {grade} — {sample['card_name']}")
        axes[row, col].axis('off')

plt.suptitle('Stage 2 Grad-CAM: Where does the condition encoder attend?', fontsize=14)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] / 'stage2_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey question: Does attention focus on card body (edges, corners, surface)')
print('or on PSA label/slab casing (shortcut)?')

In [ ]:
# Restore original forward method
stage2_model.forward = stage2_model._original_forward
print('Original forward restored')

In [ ]:
## Quantitative attention region analysis
## Divide image into regions: top strip (PSA label), card body, bottom strip
## PSA label is typically in the top ~15% of the slab image

label_region_pct = []
body_region_pct = []

n_samples = min(200, len(test_df))
sample_df = test_df.sample(n_samples, random_state=SEED)

for _, sample in sample_df.iterrows():
    img_path = PROJECT_ROOT / sample['local_image_path']
    raw_img = Image.open(img_path).convert('RGB')
    input_tensor = eval_transform(raw_img).unsqueeze(0).to(device)

    cam = gradcam.generate(input_tensor)
    cam_resized = np.array(Image.fromarray(cam).resize((224, 224), Image.BILINEAR))

    h = cam_resized.shape[0]
    label_strip = cam_resized[:int(h * 0.15), :]  ## Top 15% = PSA label
    body = cam_resized[int(h * 0.15):int(h * 0.90), :]  ## 15-90% = card body

    total_activation = cam_resized.sum()
    if total_activation > 0:
        label_region_pct.append(label_strip.sum() / total_activation * 100)
        body_region_pct.append(body.sum() / total_activation * 100)

print(f'ATTENTION REGION ANALYSIS (n={len(label_region_pct)})')
print(f'PSA label region (top 15%):  {np.mean(label_region_pct):.1f}% ± {np.std(label_region_pct):.1f}%')
print(f'Card body region (15-90%):   {np.mean(body_region_pct):.1f}% ± {np.std(body_region_pct):.1f}%')
print(f'\nv1 reference: 93.5% card body, 6.5% label')
print(f'\nIf body > 80%: model attends to card, not label (no shortcut)')
print(f'If label > 20%: possible shortcut via PSA label text')

In [ ]:
## Body masking experiment. Mask card body and check accuracy drop
## If model depends on card body, masking it should collapse accuracy
## If model uses label shortcut, masking body won't matter

def create_body_masked_transform(mask_region=(0.15, 0.90)):
    """Replace card body pixels with gray."""
    base_transform = eval_transform

    class BodyMaskTransform:
        def __call__(self, img):
            tensor = base_transform(img)
            h = tensor.shape[1]
            start = int(h * mask_region[0])
            end = int(h * mask_region[1])
            tensor[:, start:end, :] = 0  ## Zero out body region
            return tensor

    return BodyMaskTransform()

masked_transform = create_body_masked_transform()

masked_dataset = CardDataset(test_df, target_col='grade',
                             label_map=GRADE_MAP, transform=masked_transform)
masked_loader = DataLoader(masked_dataset, batch_size=CONFIG['stage2']['batch_size'],
                           shuffle=False, num_workers=2, pin_memory=True)

masked_metrics = evaluate_s2(stage2_model, stage1_model, masked_loader,
                              criterion_s2, ortho_loss_fn, device)

print(f'BODY MASKING EXPERIMENT')
print(f'Normal test accuracy:  {test_metrics_s2["accuracy"]:.1f}%')
print(f'Masked test accuracy:  {masked_metrics["accuracy"]:.1f}%')
print(f'Majority baseline:     {majority_grade_test:.1f}%')
print(f'Accuracy drop:         {test_metrics_s2["accuracy"] - masked_metrics["accuracy"]:+.1f}pp')
print(f'\nIf masked accuracy ≈ baseline: model depends on card body (good)')
print(f'If masked accuracy ≈ normal:   model uses label/casing (shortcut)')


## Section 12: Comparison: v2 vs v1

**Objective:** Systematic comparison of v2 two-stage results against v1 single-stage results. This documents the methodological improvement and justifies the architectural change.

In [ ]:
## v1 vs v2 comparison table

comparison = {
    'Metric': [
        'Dataset size',
        'Architecture',
        'Identity disentanglement',
        'Grade accuracy',
        'Majority baseline',
        'Lift over baseline',
        'Silhouette (grade)',
        'NN lift (grade)',
        'Ordinal consistency',
        'Grade MAE',
        'Adjacent error ratio',
        'Identity-condition |cos_sim|',
        'Effective dims (90%)',
        'Body attention %',
    ],
    'v1 (single-stage)': [
        '1,123',
        'Frozen ResNet50, single 256-dim head',
        'None (confounded)',
        '52.1%',
        '39.1%',
        '+13.0pp',
        '0.033',
        '~1.5x',
        'Confirmed (67+ groups)',
        '0.54',
        '87.7%',
        'N/A (no identity branch)',
        '73-85 per grade',
        '93.5%',
    ],
    'v2 (two-stage)': [
        '3,812',
        'Frozen ResNet50, Stage1 identity + Stage2 condition (orthogonal)',
        'Complete (|cos_sim| = 0.000004)',
        f'{test_metrics_s2["accuracy"]:.1f}%',
        f'{majority_grade_test:.1f}%',
        f'+{test_metrics_s2["accuracy"] - majority_grade_test:.1f}pp',
        f'{sil_condition:.4f}',
        f'{cond_lift:.3f}x',
        f'{ord_df["ordinal_consistent"].sum()}/{len(ord_df)} ({ord_df["ordinal_consistent"].mean()*100:.1f}%)',
        f'{errors.mean():.3f}',
        f'{(errors == 1).sum() / max((errors > 0).sum(), 1)*100:.1f}%',
        f'{test_metrics_s2["mean_cos_sim"]:.6f}',
        f'{eff_90_cond}',
        f'{np.mean(body_region_pct):.1f}%',
    ],
}

comp_df = pd.DataFrame(comparison)
print('VISION MODULE v1 vs v2 COMPARISON')
print(comp_df.to_string(index=False))

## Save comparison
comp_df.to_csv(CONFIG['results_dir'] / 'v1_v2_comparison.csv', index=False)

print(f'\nKEY FINDINGS')
print(f'1. v2 achieves complete identity-condition separation (v1 had none)')
print(f'2. v2 grade accuracy is slightly lower. Expected cost of removing identity shortcuts')
print(f'3. v2 ordinal consistency is tested on identity-controlled embeddings (cleaner evidence)')
print(f'4. v2 condition embedding is more concentrated (fewer effective dims = focused signal)')
print(f'5. The ~3pp accuracy drop from v1→v2 quantifies how much v1 relied on identity leakage')


## Section 13: Failure mode documentation

In [ ]:
## Document all failure modes and negative results

failure_modes = {
    'Embedding_collapse': {
        'description': 'Condition embeddings collapse to zero or constant vectors',
        'status': 'Minor. 6/3812 zero vectors (0.16%)',
        'impact': 'Negligible. Likely caused by ReLU killing all activations for edge-case images.',
        'mitigation': 'These 6 samples will have zero condition contribution in fusion. Not actionable at this scale.'
    },
    'Grade_separability': {
        'description': 'Global grade clusters not formed in condition embedding space',
        'status': 'Confirmed. Silhouette = -0.0185',
        'impact': 'Condition embeddings do not globally separate grades. Local structure exists (NN lift 1.154x, ordinal consistency 71.4%) but global separability is absent.',
        'root_cause': 'PSA 8-10 condition differences are below the resolution threshold of frozen ImageNet features at 224×224. Different cards under the same grade have vastly different visual appearances, overwhelming the subtle condition signal.',
        'mitigation': 'Future work: higher resolution input, fine-grained attention, or card-conditioned evaluation.'
    },
    'PSA9_bias': {
        'description': 'Stage 2 classifier heavily biased toward PSA 9 predictions',
        'status': 'Confirmed. PSA 9 predicted for 75.5% of test samples',
        'impact': 'PSA 8 recall = 17%, PSA 10 recall = 32%. Classifier hedges toward plurality class when uncertain.',
        'root_cause': 'Weak signal forces the model to exploit class prior. PSA 9 is the most common grade (40.2%).',
        'mitigation': 'Classifier accuracy is Auxiliary. The embedding is the primary output. Fusion module uses embeddings, not predicted grades.'
    },
    'Orthogonality_cost': {
        'description': 'Orthogonality constraint reduces grade classification accuracy vs v1',
        'status': 'Confirmed. 49.0% vs v1 52.1% (-3.1pp)',
        'impact': 'Expected and acceptable trade-off. The accuracy drop quantifies identity leakage in v1.',
        'interpretation': '~3pp of v1 grade accuracy came from card identity information, not condition. v2 removes this confound.'
    },
    'Identity_residual_in_condition': {
        'description': 'Some card_name structure visible in condition t-SNE',
        'status': 'Minor. Card clusters weakly visible but substantially reduced vs identity space',
        'impact': 'Orthogonality is measured at the embedding layer, but the shared backbone means some card information passes through before the embedding projection. Cosine similarity is 0.000004, so the embeddings themselves are orthogonal even if t-SNE picks up residual backbone structure.',
        'mitigation': 'Document as limitation. Perfect disentanglement is not theoretically possible (Locatello et al., 2019). Partial separation is sufficient for the core project claim.'
    }
}

print('FAILURE MODE DOCUMENTATION')
for code, fm in failure_modes.items():
    print(f'\n{code}: {fm["description"]}')
    print(f'Status: {fm["status"]}')
    print(f'Impact: {fm["impact"]}')

## Save
with open(CONFIG['results_dir'] / 'failure_modes.json', 'w') as f:
    json.dump(failure_modes, f, indent=2)

print(f'\nFailure modes saved to results/vision_v2/failure_modes.json')

## Section 14: Final condition-encoder assessment

In [ ]:
## Final condition-encoder verdict

print('FINAL condition-encoder ASSESSMENT')
print()
print('condition-encoder: Can a CNN trained on PSA slab images learn a latent condition')
print('embedding whose internal structure correlates with established grading')
print('dimensions and whose visual attributions localise to condition-relevant')
print('regions of the card?')
print()
print('VERDICT: PARTIALLY SUPPORTED (with stronger methodology than v1)')
print()
print('Representational criterion')
print(f'[O1] Grade classification accuracy: {test_metrics_s2["accuracy"]:.1f}% '
      f'(baseline {majority_grade_test:.1f}%, lift +{test_metrics_s2["accuracy"] - majority_grade_test:.1f}pp)')
print(f'→ Condition embedding carries grade-relevant signal above chance')
print(f'[O2] Within-card ordinal consistency: {ord_df["ordinal_consistent"].sum()}/{len(ord_df)} '
      f'({ord_df["ordinal_consistent"].mean()*100:.1f}%)')
print(f'→ Embedding distances respect grade ordering for majority of Pokémon')
print(f'[O3] NN grade lift: {cond_lift:.3f}x (identity-controlled)')
print(f'→ Local grade structure present in condition space')
print(f'[O4] Global grade separability: silhouette = {sil_condition:.4f}')
print(f'→ Grades not globally separable. Weak signal, high within-grade variation')
print()
print('Interpretive criterion')
print(f'[O5] Grad-CAM body attention: {np.mean(body_region_pct):.1f}%')
print(f'→ Model attends to card body, not PSA label (no shortcut)')
print(f'[O5] Body masking accuracy drop: '
      f'{test_metrics_s2["accuracy"] - masked_metrics["accuracy"]:+.1f}pp')
print(f'→ Model depends on card body content')
print()
print('Separation criterion')
print(f'[O6] Identity-condition |cos_sim|: {test_metrics_s2["mean_cos_sim"]:.6f}')
print(f'→ Complete orthogonality achieved')
print(f'[O6] Stage 1 identity accuracy: 91.3% (card_name classification)')
print(f'→ Identity subspace well-defined before condition training')
print()
print('Methodological advancement over v1')
print(f'• v1 could not distinguish condition signal from identity leakage')
print(f'• v2 enforces separation via orthogonality constraint (Locatello et al., 2019)')
print(f'• v2 accuracy drop (-3.1pp vs v1) quantifies identity leakage in v1')
print(f'• v2 provides cleaner evidence for condition-encoder even with weaker raw accuracy')
print()
print('Limitations')
print(f'• PSA 8-10 condition differences may be below frozen ImageNet resolution')
print(f'• Condition embedding is concentrated in ~5 dimensions (limited signal)')
print(f'• PSA 9 classification bias (75.5% of predictions)')
print(f'• Different actual cards under same Pokémon name remain a confound')
print(f'within the condition space (partially addressed, not fully resolved)')
print()
print('Downstream expectation')
print(f'Condition embeddings carry a weak but genuine visual signal.')
print(f'The fusion module should not expect strong independent price prediction')
print(f'from vision (condition) alone. The value of the condition embedding is marginal signal')
print(f'that the market module cannot access physical card quality.')
print()

In [ ]:
## Save all final artefacts
rq2_assessment = {
    'verdict': 'PARTIALLY SUPPORTED',
    'grade_accuracy': test_metrics_s2['accuracy'],
    'baseline': majority_grade_test,
    'lift': test_metrics_s2['accuracy'] - majority_grade_test,
    'silhouette_condition': sil_condition,
    'silhouette_identity_grade': sil_identity_grade,
    'silhouette_identity_name': sil_identity_name,
    'nn_lift_condition': cond_lift,
    'nn_lift_identity': id_lift,
    'ordinal_consistency': f'{ord_df["ordinal_consistent"].sum()}/{len(ord_df)}',
    'orthogonality_cos_sim': test_metrics_s2['mean_cos_sim'],
    'body_attention_pct': float(np.mean(body_region_pct)),
    'masked_accuracy': masked_metrics['accuracy'],
    'effective_dims_90': int(eff_90_cond),
    'effective_dims_95': int(eff_95_cond),
    'grade_mae': float(errors.mean()),
    'adjacent_error_ratio': float((errors == 1).sum() / max((errors > 0).sum(), 1)),
    'stage1_accuracy': 91.3,
    'v1_accuracy': 52.1,
    'zero_vectors': 6,
}

with open(CONFIG['results_dir'] / 'rq2_final_assessment.json', 'w') as f:
  json.dump(rq2_assessment, f, indent=2, default=lambda x: float(x) if hasattr(x, 'item') else str(x))
print('Final condition-encoder assessment saved to results/vision_v2/rq2_final_assessment.json')
print(f'\nSaved artefacts:')
print(f'Models: stage1_identity_best.pth, stage2_condition_best.pth')
print(f'Embeddings: visual_embeddings_v2.parquet (condition, for fusion)')
print(f'identity_embeddings_v2.parquet (diagnostic)')
print(f'Metrics: stage1_metrics.json, stage2_metrics.json')
print(f'Assessment: rq2_final_assessment.json, failure_modes.json')
print(f'Comparison: v1_v2_comparison.csv')
print(f'Figures:  {len(list(CONFIG["figures_dir"].glob("*.png")))} PNG files')
print(f'\nVISION MODULE v2 COMPLETE')